In [3]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

from openai import AsyncOpenAI

import os

In [6]:
api_key = os.getenv("CHATNS_API_KEY")

In [7]:
model = OpenAIChatModel(
    model_name="gpt-4.1-mini",
    provider=OpenAIProvider(
        openai_client=AsyncOpenAI(
            api_key=api_key,
            base_url="https://api.openai.com/v1",
        ),
    ),
)

In [33]:
from pathlib import Path
import json

cache_dir = Path("../../.cache")

cache_file = cache_dir / "db_metadata.json"

if cache_file.exists():
    with open(cache_file, "r") as f:
        cached_metadata = json.load(f)

stations_file = cache_dir / "stations.json"
if stations_file.exists():
    with open(stations_file, "r") as f:
        cached_stations = json.load(f)

In [ ]:
available_tables = ""
for table in cached_metadata.values():
    available_tables += f"{table['name']}: {table['comment']}\n"

print(available_tables)

dimdatum: Datumdimensie met kalenderkenmerken voor rapportages en fact-koppelingen.
dimdienstregelpunt: Dimensietabel met alle dienstregelpunten (stations, haltes en knooppunten) op het NS-netwerk. Wordt gebruikt als locatiesleutel in factabellen voor rapportage per regio.
dimlocatietype: Dimensietabel met de type locaties waarop een incident kan plaatsvinden binnen het NS-domein.
dimmeldingssoort: Dimensietabel met de categorisering van meldingen in meldingsoort, hoofdsoort, en ABC-categorie conform
landelijke OV-afspraken.
dimtijd: Tijddimensie met uurblokken, dagdelen en piekuurindicators voor verkeers- en logistieke analyses.
dimtreinnummer_treinserie: Dimensietabel met de combinatie van treinnummer en treinserie zoals geregistreerd op het moment van het incident. Wordt gebruikt om incidenten te analyseren per trein en lijn. Speciale waarde -3 voor de surrogaatsleutel als het incident niet op een trein plaatsvond.
factincidentmkns: Facttabel met alle door de Meldkamer NS afgehandel

In [42]:
cached_stations

[{'code': 'Ht',
  'naam': "'s-Hertogenbosch",
  'regio_rsv': 'RSV Zuid',
  'regio_ssvo': 'Oost-Brabant en Noord-Limburg'},
 {'code': 'Hto',
  'naam': "'s-Hertogenbosch Oost",
  'regio_rsv': 'RSV Zuid',
  'regio_ssvo': 'Oost-Brabant en Noord-Limburg'},
 {'code': 'Hde',
  'naam': "'t Harde",
  'regio_rsv': 'RSV Noord-Oost',
  'regio_ssvo': 'PE Noord'},
 {'code': '-1',
  'naam': '<Niet gevonden>',
  'regio_rsv': 'Unknown',
  'regio_ssvo': 'Unknown'},
 {'code': '-3',
  'naam': '<Niet van toepassing>',
  'regio_rsv': 'Unknown',
  'regio_ssvo': 'Unknown'},
 {'code': '-2',
  'naam': '<Ontbrekend>',
  'regio_rsv': 'Unknown',
  'regio_ssvo': 'Unknown'},
 {'code': 'Atn',
  'naam': 'Aalten',
  'regio_rsv': 'RSV Noord-Oost',
  'regio_ssvo': 'Arnhem-Nijmegen'},
 {'code': 'Ac',
  'naam': 'Abcoude',
  'regio_rsv': 'RSV Randstad Noord',
  'regio_ssvo': 'Amsterdam'},
 {'code': 'Akm',
  'naam': 'Akkrum',
  'regio_rsv': 'RSV Noord-Oost',
  'regio_ssvo': 'PE Noord'},
 {'code': 'Amr',
  'naam': 'Alkmaar',


In [52]:
available_stations = ""
for station in cached_stations:
    if station["code"] not in ['-1', '-2', '-3']:
        available_stations += f"{station['naam']} ({station['code']})\n"

print(available_stations)

's-Hertogenbosch (Ht)
's-Hertogenbosch Oost (Hto)
't Harde (Hde)
Aalten (Atn)
Abcoude (Ac)
Akkrum (Akm)
Alkmaar (Amr)
Alkmaar Noord (Amrn)
Almelo (Aml)
Almelo de Riet (Amri)
Almere Buiten (Almb)
Almere Centrum (Alm)
Almere Muziekwijk (Almm)
Almere Oostvaarders (Almo)
Almere Parkwijk (Almp)
Almere Poort (Ampo)
Alphen aan den Rijn (Apn)
Amersfoort Centraal (Amf)
Amersfoort Schothorst (Amfs)
Amersfoort Vathorst (Avat)
Amsterdam Amstel (Asa)
Amsterdam Arena (Asdar)
Amsterdam Bijlmer ArenA (Asb)
Amsterdam Centraal (Asd)
Amsterdam Holendrecht (Ashd)
Amsterdam Lelylaan (Asdl)
Amsterdam Muiderpoort (Asdm)
Amsterdam RAI (Rai)
Amsterdam Science Park (Assp)
Amsterdam Sloterdijk (Ass)
Amsterdam Zuid (Asdz)
Anna Paulowna (Ana)
Apeldoorn (Apd)
Apeldoorn De Maten (Apdm)
Apeldoorn Osseveld (Apdo)
Appingedam (Apg)
Arkel (Akl)
Arnemuiden (Arn)
Arnhem Centraal (Ah)
Arnhem Presikhaaf (Ahpr)
Arnhem Velperpoort (Ahp)
Arnhem Zuid (Ahz)
Assen (Asn)
Baarn (Brn)
Bad Nieuweschans (Nsch)
Baflo (Bf)
Barendrecht (B

In [53]:
CLARIFIER_INSTRUCTIONS = f"""
Your goal is to clarify user questions about incident data so 
that they can be handed off to the SQL agent.

You only accept questions that are related to data that is available in the 
database. If the user asks something about data that is not available, 
you should respond with a polite message indicating that the data 
is not available and give an overview of what data you have available.

## Goal
Make sure the user question is clear and complete enough to be handed 
off to the SQL agent. This means you need to ensure that the following 
information is present:
- A valid date range (from_date and to_date)
- Valid station names when stations are mentioned
- Any additional filters that user clearly asks for

## Available tables with descriptions:
{available_tables}

## Available stations:
{available_stations}

## Behavior
- Ask exactly one concise follow-up question per turn when information is missing.
- Never generate SQL.
- Keep wording short and practical.

## Completion Signal
When the request is ready for structured extraction, end your response with the exact token: [READY]
""".strip()

conversational_agent = Agent(
    name="ReactClarifierAgent",
    model=model,
    system_prompt=CLARIFIER_INSTRUCTIONS,
    output_type=str,
)

In [58]:
import asyncio

In [ ]:
async def run_agent(
        agent: Agent,
        user_prompt: str,
        message_history=None
    ) -> AgentRunResult:

    if message_history is None:
        message_history = []

    result = await agent.run(
        user_prompt,
        message_history=message_history,
        # output_type=RAGResponse
    )

    return result

In [ ]:
clarifier = conversational_agent
history = []
MAX_CLARIFICATION_TURNS = 2
READY_MARKER = "[READY]"

while True:
    try:
        user_line = input("User 👤: ").strip()
        if not user_line:
            continue
        if user_line.lower() in ["exit", "quit"]:
            print("Closing system session. Goodbye!")
            break
        
        # agent_run = asyncio.run(
        #     run_agent_stream(agent, prompt, message_history=message_history)
        # )
        # if not agent_run:
        #     continue

        # message_history = agent_run.all_messages()
        # output = agent_run.output
        # print()

        for _ in range(MAX_CLARIFICATION_TURNS):
            print("\nRefinement Agent 🤖 is processing your request...\n")
            clarification_result = clarifier.run_sync(user_line, message_history=history)
            history = clarification_result.all_messages()
            print(history)

            response_text = clarification_result.output.strip()
            print(response_text)
            if READY_MARKER in response_text:
                clean_text = response_text.replace(READY_MARKER, "").strip()
                if clean_text:
                    print(f"\nRefinement Agent 🤖: {clean_text}\n")
                    break

    except RuntimeError as e:
        print(f"\n[System] {e}\n")
    except (KeyboardInterrupt, EOFError):
        print("\nTerminal interrupt encountered. Closing cleanly.")
        break


Refinement Agent 🤖 is processing your request...


[System] This event loop is already running



/var/folders/f7/3z74v0vx4sb57gfvzq84q0z80000gn/T/ipykernel_21249/3466318169.py:40: RuntimeWarning: coroutine 'AbstractAgent.run' was never awaited
  print(f"\n[System] {e}\n")


Closing system session. Goodbye!
